## Create the list of Face Videos

In [1]:
import os
import pickle
import gzip
from pathlib import Path

# === CONFIGURATION ===
input_dir = Path("face_data/")  # Change this to your raw video directory
output_path = Path("./")
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / "face_files_list.pkl.gz"  # This matches your main code
video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

# === SCAN VIDEO FILES ===
video_paths = [
    str(file.resolve()) for file in input_dir.rglob("*")
    if file.suffix.lower() in video_exts
]

print(f"Found {len(video_paths)} video files.")

# === SAVE TO .pkl.gz ===
with gzip.open(output_file, 'wb') as f:
    pickle.dump(video_paths, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved video path list to: {output_file}")


Found 10 video files.
Saved video path list to: face_files_list.pkl.gz


## Extract Face Features using DINOv2

In [2]:
import os
import csv
import time
import json
import gzip
import glob
import pickle
import torch
import numpy as np
import torch.nn as nn
from PIL import Image
from pathlib import Path
from torchvision import transforms
from decord import VideoReader, cpu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [3]:
def get_mp4_files(directory):
    if not os.path.exists(directory):
        raise FileNotFoundError(f'Directory not found: {directory}')
    return [os.path.abspath(file) for file in glob.glob(os.path.join(directory, '*.mp4'))]

def load_file(filename):
    with gzip.open(filename, "rb") as f:
        return pickle.load(f)

def is_string_in_file(file_path, target_string):
    try:
        with Path(file_path).open("r") as f:
            return any(target_string in line for line in f)
    except Exception as e:
        print(f"Error: {e}")
        return False


In [4]:
def get_dino_finetuned_downloaded(dino_path):
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14_reg', pretrained=False)
    pretrained = torch.load(dino_path, map_location=device)

    new_state_dict = {}
    for key, value in pretrained['teacher'].items():
        if 'dino_head' in key:
            continue
        else:
            new_key = key.replace('backbone.', '')
            new_state_dict[new_key] = value

    model.pos_embed = nn.Parameter(torch.zeros(1, 257, 384))
    model.load_state_dict(new_state_dict, strict=True)
    model.to(device)
    return model

In [5]:
def preprocess_frame(frame):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    image = Image.fromarray(frame)
    return transform(image)[:3]


In [6]:
def video_to_embeddings(video_path, output_folder, dino_path, batch_size=128):
    try:
        vr = VideoReader(video_path, width=224, height=224)
    except Exception as e:
        print(f'Failed to load video: {video_path} | Error: {e}')
        return

    total_frames = len(vr)
    all_embeddings = []

    for idx in range(0, total_frames, batch_size):
        batch_frames = vr.get_batch(range(idx, min(idx + batch_size, total_frames))).asnumpy()
        batch_tensors = torch.stack([preprocess_frame(frame) for frame in batch_frames]).to(device)

        with torch.no_grad():
            batch_embeddings = model(batch_tensors).cpu().numpy()
        
        all_embeddings.append(batch_embeddings)

    embeddings = np.concatenate(all_embeddings, axis=0)
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    np.save(f"{output_folder}/{video_name}.npy", embeddings)


In [9]:
index=0
time_limit=36000
batch_size=100
files_list="face_files_list.pkl.gz"
output_folder="face_features_data"
dino_path="face_dinov2_checkpoint.pth"


start_time = time.time()
fixed_list = load_file(files_list)
video_batches = [fixed_list[i:i + batch_size] for i in range(0, len(fixed_list), batch_size)]

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

model = get_dino_finetuned_downloaded(dino_path)

for video_path in video_batches[index]:
    current_time = time.time()
    if current_time - start_time > time_limit:
        print("Time limit reached. Stopping execution.")
        break

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    np_path = f"{output_folder}/{video_name}.npy"

    if os.path.exists(np_path):
        continue
    else:
        video_to_embeddings(video_path, output_folder, dino_path, batch_size=512)

Using cache found in /home/ashishu23/.cache/torch/hub/facebookresearch_dinov2_main
/home/ashishu23/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:37: UserWarning: xFormers is available (Block)
  except ImportError:


## Create the list of Hands Videos

In [10]:
import os
import pickle
import gzip
from pathlib import Path

# === CONFIGURATION ===
input_dir = Path("hands_data/")  # Change this to your raw video directory
output_path = Path("./")
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / "hands_files_list.pkl.gz"  # This matches your main code
video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

# === SCAN VIDEO FILES ===
video_paths = [
    str(file.resolve()) for file in input_dir.rglob("*")
    if file.suffix.lower() in video_exts
]

print(f"Found {len(video_paths)} video files.")

# === SAVE TO .pkl.gz ===
with gzip.open(output_file, 'wb') as f:
    pickle.dump(video_paths, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved video path list to: {output_file}")


Found 20 video files.
Saved video path list to: hands_files_list.pkl.gz


## Extract Hands Features using DINOv2

In [11]:
index=0
time_limit=36000
batch_size=100
files_list="hands_files_list.pkl.gz"
output_folder="hands_features_data"
dino_path="hands_dinov2_checkpoint.pth"


start_time = time.time()
fixed_list = load_file(files_list)
video_batches = [fixed_list[i:i + batch_size] for i in range(0, len(fixed_list), batch_size)]

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

model = get_dino_finetuned_downloaded(dino_path)

for video_path in video_batches[index]:
    current_time = time.time()
    if current_time - start_time > time_limit:
        print("Time limit reached. Stopping execution.")
        break

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    np_path = f"{output_folder}/{video_name}.npy"

    if os.path.exists(np_path):
        continue
    else:
        video_to_embeddings(video_path, output_folder, dino_path, batch_size=512)

Using cache found in /home/ashishu23/.cache/torch/hub/facebookresearch_dinov2_main
